# 01 — Baseline U-Net Training

This notebook trains a baseline U-Net model for accelerated MRI reconstruction on the FastMRI single-coil knee dataset.

The acquisition is simulated by applying a **regular Cartesian undersampling mask at acceleration factor 4 (AF4)** to fully-sampled k-space data. The network learns to reconstruct the aliased zero-filled image back to the fully-sampled ground truth.

Trained weights are saved to `../weights/unet_baseline.pth` for use in the robustness experiments (`02_robustness_experiments.ipynb`).

## 1. Imports and Setup

All model definitions, data utilities, metrics, and training functions are imported from `src/`. The random seed is fixed at 42 for reproducibility.

In [ ]:
import os
import sys
import random
import numpy as np
import torch
import matplotlib.pyplot as plt

# Make src/ importable when running from notebooks/
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))

from src.model   import define_model
from src.dataset import DatasetFastMRI
from src.metrics import calculate_psnr_single, calculate_ssim_single
from src.utils   import mkdir, total_loss
from src.train   import train_one_epoch, validate, validate_one_step
from torch.utils.data import DataLoader
from functools import partial

# Reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

## 2. Parameters

All experiment hyperparameters are defined in a single cell so they are easy to modify.

In [ ]:
model_name       = 'unet'
mask_name        = 'fMRI_Reg_AF4_CF0.08_PE320'
train_path       = '../data/fastmri_tiny/train'
test_path        = '../data/fastmri_tiny/val'
masks_root       = os.path.abspath('../masks/fastmri')
train_batch_size = 4
test_batch_size  = 1
total_epochs     = 100
lr               = 1e-4
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

in_channels  = 2
out_channels = 2

print(f'Device     : {device}')
print(f'Model      : {model_name}')
print(f'Mask       : {mask_name}')
print(f'Masks root : {masks_root}')


## 3. Model Instantiation

The U-Net takes a 2-channel input (real and imaginary parts of the undersampled complex image) and produces a 2-channel output (predicted real and imaginary parts of the reconstruction).

In [ ]:
model_type = define_model(model_name)
model      = model_type(in_channels=in_channels, out_channels=out_channels).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: {model_name}  |  Trainable parameters: {n_params:,}')

## 4. Dataloaders

`DatasetFastMRI` loads each `.h5` file, normalises the complex image, applies the k-space undersampling mask, and returns a `(undersampled, ground_truth)` pair as 2-channel float32 tensors.

In [ ]:
train_dataset = DatasetFastMRI(data_path_src=train_path, mask_name=mask_name, masks_root=masks_root)
test_dataset  = DatasetFastMRI(data_path_src=test_path,  mask_name=mask_name, masks_root=masks_root)

train_loader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True,  drop_last=True)
test_loader  = DataLoader(test_dataset,  batch_size=test_batch_size,  shuffle=False, drop_last=False)

print(f'Train samples: {len(train_dataset)}  |  batches: {len(train_loader)}')
print(f'Test  samples: {len(test_dataset)}   |  batches: {len(test_loader)}')


## 5. Loss Function and Optimiser

The combined loss is a weighted sum of an L1 image-domain term and an L1 frequency-domain term (computed on the FFT of the predictions and targets). The Adam optimiser is used with a learning rate of 1e-4.

In [ ]:
loss_fn = partial(
    total_loss,
    loss_image_weight=15,
    loss_image_type='l1',
    loss_freq_weight=0.1,
    loss_freq_type='l1',
    device=device,
)

optimizer = torch.optim.Adam(model.parameters(), lr=lr)

print(f'Loss : image L1 (w=15) + freq L1 (w=0.1)')
print(f'Optim: Adam  lr={lr}')

## 6. Training Loop

The model is trained for `total_epochs` epochs. Loss is recorded at every step and averaged per epoch. Both curves are plotted after training completes.

In [ ]:
train_losses_per_epoch = []
train_losses_per_step  = []

print('Starting training...')
for epoch in range(total_epochs):
    epoch_loss, batch_losses = train_one_epoch(
        model, train_loader, optimizer, loss_fn, epoch, total_epochs
    )
    train_losses_per_epoch.append(epoch_loss)
    train_losses_per_step.extend(batch_losses)

print('Training complete.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(range(1, total_epochs + 1), train_losses_per_epoch, marker='o', markersize=3)
axes[0].set_title('Training Loss per Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True)

axes[1].plot(range(1, len(train_losses_per_step) + 1), train_losses_per_step, linewidth=0.8)
axes[1].set_title('Training Loss per Step')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Loss')
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 7. Save Weights

The trained model weights are saved to `../weights/unet_baseline.pth`. This file is loaded by `02_robustness_experiments.ipynb` to evaluate the model under distribution shifts without retraining.

In [ ]:
weights_dir  = '../weights'
weights_path = os.path.join(weights_dir, 'unet_baseline.pth')

mkdir(weights_dir)
torch.save(model.state_dict(), weights_path)
print(f'Weights saved to {weights_path}')

## 8. Evaluation

The model is evaluated on the held-out test set. PSNR and SSIM are computed for both the model prediction and the zero-filled (ZF) baseline to quantify reconstruction quality improvement.

In [ ]:
metrics_dict = validate(model, test_loader)

val_psnr    = np.mean(metrics_dict['psnr'])
val_ssim    = np.mean(metrics_dict['ssim'])
val_psnr_zf = np.mean(metrics_dict['psnr_zf'])
val_ssim_zf = np.mean(metrics_dict['ssim_zf'])

print(f'Reconstruction  — PSNR: {val_psnr:.4f} dB  |  SSIM: {val_ssim:.4f}')
print(f'Zero-filled ZF  — PSNR: {val_psnr_zf:.4f} dB  |  SSIM: {val_ssim_zf:.4f}')
print(f'PSNR improvement: {val_psnr - val_psnr_zf:+.4f} dB')
print(f'SSIM improvement: {val_ssim - val_ssim_zf:+.4f}')

In [ ]:
validate_one_step(model, test_loader)